# Feature Engineering

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')


In [4]:
df = pd.read_csv('HHS_cleaned.csv')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 17 columns):
 #   Column                                           Non-Null Count  Dtype  
---  ------                                           --------------  -----  
 0   Date                                             720 non-null    object 
 1   Children apprehended and placed in CBP custody*  720 non-null    float64
 2   Children in CBP custody                          720 non-null    float64
 3   Children transferred out of CBP custody          720 non-null    float64
 4   Children discharged from HHS Care                720 non-null    float64
 5   Children in HHS Care                             720 non-null    int64  
 6   Month                                            720 non-null    int64  
 7   Year                                             720 non-null    int64  
 8   Rolling_Mean_7                                   714 non-null    float64
 9   Noise                           

In [6]:
df['Date'] = pd.to_datetime(df['Date'])

In [7]:
df = df.sort_values('Date').reset_index(drop=True)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 17 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Date                                             720 non-null    datetime64[ns]
 1   Children apprehended and placed in CBP custody*  720 non-null    float64       
 2   Children in CBP custody                          720 non-null    float64       
 3   Children transferred out of CBP custody          720 non-null    float64       
 4   Children discharged from HHS Care                720 non-null    float64       
 5   Children in HHS Care                             720 non-null    int64         
 6   Month                                            720 non-null    int64         
 7   Year                                             720 non-null    int64         
 8   Rolling_Mean_7                          

In [9]:
print(df.columns.tolist())

['Date', 'Children apprehended and placed in CBP custody*', 'Children in CBP custody', 'Children transferred out of CBP custody', 'Children discharged from HHS Care', 'Children in HHS Care ', 'Month', 'Year', 'Rolling_Mean_7', 'Noise', 'Lag_1', 'Lag_7', 'Date_diff', 'Day', 'Lag_30', 'Rolling_mean_7_forecast', 'Rolling_STD_7']


In [10]:
# creating remaining feature

df['Lag_3']  = df['Children in HHS Care '].shift(3)
df['Lag_14'] = df['Children in HHS Care '].shift(14)

In [11]:
df['Rolling_mean_14_forecast'] = df['Children in HHS Care '].shift(1).rolling(14).mean()

In [12]:
df['Day_of_week'] = df['Date'].dt.dayofweek
df['Week_of_Year'] = df['Date'].dt.isocalendar().week.astype(int)


In [13]:
df['Rolling_Var_7_forecast'] = (
    df['Children in HHS Care '].shift(1).rolling(7).var()
)

In [14]:
df['Rolling_Var_14_forecast'] = (
    df['Children in HHS Care '].shift(1).rolling(14).var()
)

In [15]:
# creating the flow based signal that project asked

df['Net_Pressure'] = (
    df['Children transferred out of CBP custody']
    - df['Children discharged from HHS Care']
)

In [16]:
df['Net_Pressure_lag_1'] = df['Net_Pressure'].shift(1)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 720 entries, 0 to 719
Data columns (total 26 columns):
 #   Column                                           Non-Null Count  Dtype         
---  ------                                           --------------  -----         
 0   Date                                             720 non-null    datetime64[ns]
 1   Children apprehended and placed in CBP custody*  720 non-null    float64       
 2   Children in CBP custody                          720 non-null    float64       
 3   Children transferred out of CBP custody          720 non-null    float64       
 4   Children discharged from HHS Care                720 non-null    float64       
 5   Children in HHS Care                             720 non-null    int64         
 6   Month                                            720 non-null    int64         
 7   Year                                             720 non-null    int64         
 8   Rolling_Mean_7                          

In [18]:
df[[
    'Date',
    'Children in HHS Care ',
    'Lag_1',
    'Lag_7',
    'Lag_14',
    'Rolling_mean_7_forecast',
    'Rolling_mean_14_forecast',
    'Rolling_Var_7_forecast',
    'Rolling_Var_14_forecast',
    'Net_Pressure',
    'Day_of_week',
    'Month',
    'Week_of_Year'
]].head(20)

,Date,Children in HHS Care,Lag_1,Lag_7,Lag_14,Rolling_mean_7_forecast,Rolling_mean_14_forecast,Rolling_Var_7_forecast,Rolling_Var_14_forecast,Net_Pressure,Day_of_week,Month,Week_of_Year
0,2023-01-12,6566,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-402.0,3,1,2
1,2023-01-22,7122,6566.0,NaN,NaN,NaN,NaN,NaN,NaN,-188.0,6,1,3
2,2023-01-23,7280,7122.0,NaN,NaN,NaN,NaN,NaN,NaN,-142.0,0,1,4
3,2023-01-24,7433,7280.0,NaN,NaN,NaN,NaN,NaN,NaN,-128.0,1,1,4
4,2023-01-25,7538,7433.0,NaN,NaN,NaN,NaN,NaN,NaN,-139.0,2,1,4
5,2023-01-29,7472,7538.0,NaN,NaN,NaN,NaN,NaN,NaN,-292.0,6,1,4
6,2023-01-30,7743,7472.0,NaN,NaN,NaN,NaN,NaN,NaN,-167.0,0,1,5
7,2023-01-31,7803,7743.0,6566.0,NaN,7307.714286,NaN,145098.238095,NaN,-122.0,1,1,5
8,2023-02-01,7903,7803.0,7122.0,NaN,7484.428571,NaN,57860.285714,NaN,-204.0,2,2,5
9,2023-02-02,7879,7903.0,7280.0,NaN,7596.000000,NaN,50645.333333,NaN,-275.0,3,2,5


In [19]:
feature = [
    'Lag_1', "Lag_7", 'Lag_14',
    'Rolling_mean_7_forecast', 'Rolling_mean_14_forecast',
    'Rolling_Var_7_forecast', 'Rolling_Var_14_forecast',
    'Day_of_week', 'Month', 'Week_of_Year',
    'Net_Pressure'
]

In [20]:
x = df[feature]
y = df['Children in HHS Care ']

In [21]:
x.isnull().sum()

Lag_1                        1
Lag_7                        7
Lag_14                      14
Rolling_mean_7_forecast      7
Rolling_mean_14_forecast    14
Rolling_Var_7_forecast       7
Rolling_Var_14_forecast     14
Day_of_week                  0
Month                        0
Week_of_Year                 0
Net_Pressure                 0
dtype: int64

In [22]:
model_df = df[['Date'] + feature + ['Children in HHS Care ']].dropna()

In [23]:
model_df = model_df.sort_values('Date')

In [24]:
model_df.head()

,Date,Lag_1,Lag_7,Lag_14,Rolling_mean_7_forecast,Rolling_mean_14_forecast,Rolling_Var_7_forecast,Rolling_Var_14_forecast,Day_of_week,Month,Week_of_Year,Net_Pressure,Children in HHS Care
14,2023-02-09,7915.0,7803.0,6566.0,7808.714286,7558.214286,14128.904762,141066.642857,3,2,6,-192.0,7908
15,2023-02-12,7908.0,7903.0,7122.0,7823.714286,7654.071429,15503.904762,64852.840659,6,2,6,-144.0,7434
16,2023-02-13,7434.0,7879.0,7280.0,7756.714286,7676.357143,34531.904762,46266.554945,0,2,7,-72.0,7483
17,2023-02-14,7483.0,7586.0,7433.0,7700.142857,7690.857143,40792.476190,36831.516484,1,2,7,-3.0,7794
18,2023-02-15,7794.0,7720.0,7538.0,7729.857143,7716.642857,39059.142857,31819.170330,2,2,7,-118.0,7869


In [25]:
model_df.shape

(706, 13)

In [26]:
print('start', model_df['Date'].min())
print('end', model_df['Date'].max())
print('rows', len(model_df))

start 2023-02-09 00:00:00
end 2025-12-21 00:00:00
rows 706


# Time-Based split (80-20)

In [27]:
x = model_df[feature]
y = model_df['Children in HHS Care ']

In [28]:
# chronological train-test split
split_index = int(len(model_df) * 0.8 )

train_df = model_df.iloc[:split_index]
test_df = model_df.iloc[split_index:]

In [29]:
print('Train shape', train_df.shape)
print('Test shape', test_df.shape)

print('\n Train period')
print(train_df['Date'].min(),'->', train_df['Date'].max())

print('\n test period')
print(test_df['Date'].min(), '->', test_df['Date'].max())

Train shape (564, 13)
Test shape (142, 13)

 Train period
2023-02-09 00:00:00 -> 2025-05-15 00:00:00

 test period
2025-05-18 00:00:00 -> 2025-12-21 00:00:00


In [30]:
X_train = train_df[feature]
Y_train = train_df['Children in HHS Care ']

X_test = test_df[feature]
Y_test = test_df['Children in HHS Care ']

In [31]:
print('x train ', X_train.shape)
print('y train ', Y_train.shape)

print('x test ', X_test.shape)
print('y test ', Y_test.shape)

x train  (564, 11)
y train  (564,)
x test  (142, 11)
y test  (142,)


Walk-forward validation

In [32]:
initial_train_size = 400
validation_size = 14

folds = []

start = initial_train_size

while start + validation_size <= len(train_df) :
    train_fold = train_df.iloc[:start]
    validation_fold = train_df.iloc[start:start + validation_size]

    folds.append((train_fold, validation_fold))

    start += validation_size

print("Number of folds", len(folds))

Number of folds 11


Naive baseline for 1 step walk-forward validation

In [33]:
from sklearn.metrics import mean_absolute_error

naive_mae= []
naive_results = []

for fold_number, (train_fold, validation_fold) in enumerate (folds, start = 1): 

    # Get the target value
    train_values = train_fold['Children in HHS Care '].values
    validation_values = validation_fold['Children in HHS Care '].values

    # 1 step-ahead Naive prediction
    prediction = []
    previous_value = train_values[-1]

    for actual_value in validation_values:
        # predict using previous observed value 
        prediction.append(previous_value)

        # after observing the actual value, use it for next prediction
        previous_value = actual_value

    # calculating MAE for this fold
    mae = mean_absolute_error(validation_values, prediction)
    #store MAE
    naive_mae.append(mae)

    # store everything needed for later metrics
    naive_results.append({
        'fold' : fold_number,
        'actual' : validation_values,
        'prediction' : prediction,
        'mae' : mae
    })


    print(f'Folds{fold_number} MAE : {mae:.2f}')

# avarage MAE of all folds 
avarege_naive_mae = sum(naive_mae) / len(naive_mae)
print('\nAvarage Naive MAE', round(avarege_naive_mae, 2))


Folds1 MAE : 66.79
Folds2 MAE : 65.14
Folds3 MAE : 58.57
Folds4 MAE : 65.93
Folds5 MAE : 87.71
Folds6 MAE : 104.36
Folds7 MAE : 133.50
Folds8 MAE : 37.29
Folds9 MAE : 12.36
Folds10 MAE : 10.14
Folds11 MAE : 5.07

Avarage Naive MAE 58.81


In [34]:
def moving_average_forcasting(train_values, validation_values, window = 7):
    history = list(train_values)
    predictions = []

    for actual in validation_values:
        predictions.append(sum(history[-window:]) / window)
        history.append(actual)
    return predictions

In [35]:
moving_avg_mae = []

for train_fold, validation_fold in folds:

    train_values = train_fold['Children in HHS Care '].values
    validation_values = validation_fold['Children in HHS Care ']

    prediction = moving_average_forcasting(
        train_values,
        validation_values,
        window=7
    )

    mae = mean_absolute_error(validation_values, prediction)
    moving_avg_mae.append(mae)

# Average MAE
average_moving_avg_mae = sum(moving_avg_mae) / len(moving_avg_mae)
print("Average of moving average mae", round(average_moving_avg_mae, 2))

Average of moving average mae 166.19


In [36]:
for i,mae in enumerate(moving_avg_mae, start=1):
    print(f'Fold{i} MAE : {mae:.2f}')

Fold1 MAE : 80.82
Fold2 MAE : 65.24
Fold3 MAE : 85.73
Fold4 MAE : 172.83
Fold5 MAE : 173.40
Fold6 MAE : 457.91
Fold7 MAE : 511.92
Fold8 MAE : 172.38
Fold9 MAE : 58.08
Fold10 MAE : 37.12
Fold11 MAE : 12.61


In [37]:
horizons = [1,7,14]
# I've defined the multi-horizon, now elavualate those three 

In [38]:
multi_horizon_naive = {}

for horizon in horizons :
    horizon_mae = []

    for train_fold, validation_fold in folds:
        train_values = train_fold['Children in HHS Care '].values
        validation_values = validation_fold['Children in HHS Care '].values

        # Make enough validation observation exist
        actual = validation_values[:horizon]

        # Naive forcast: use last training value
        prediction = [train_values[-1]] * horizon

        mae = mean_absolute_error(actual,prediction)
        horizon_mae.append(mae)

    multi_horizon_naive[horizon] = sum(horizon_mae) / len(horizon_mae)

for horizon,mae in multi_horizon_naive.items():
    print(f'{horizon}--Day of horizon, MAE:{mae:.2f}')

1--Day of horizon, MAE:79.18
7--Day of horizon, MAE:205.39
14--Day of horizon, MAE:313.74


In [39]:
# for lag 1,,7,14

lag_naive_result = {}

for lag in [1,7,14]:
    lag_mae = []

    for train_fold, validation_fold in folds:
        train_values = train_fold['Children in HHS Care '].values
        validation_values = validation_fold['Children in HHS Care '].values

        predictions = []

        # need enough history to use lag the selected lag 
        history = list(train_values)

        for actual in validation_values :

            # Predict using the value 'lag' observation ago
            prediction = history[-lag]
            predictions.append(prediction)

            # add actual valua to the history
            history.append(actual)

        mae = mean_absolute_error(validation_values, predictions)
        lag_mae.append(mae)

    lag_naive_result[lag] = sum(lag_mae) / len(lag_mae)

for lag, mae in lag_naive_result.items():
    print(f'lag{lag} Lag MAE : {mae:.2f}')

lag1 Lag MAE : 58.81
lag7 Lag MAE : 287.44
lag14 Lag MAE : 522.34


# Statistical Model

In [40]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(Y_train)

print(f'ADF statistics: {adf_result[0]}')
print(f'p-value: {adf_result[1]}')

ADF statistics: -0.941258922459002
p-value: 0.7740819008816717


In [41]:
# first order differencing 
Y_train_diff = Y_train.diff().dropna()

# ADF test after differencing
adf_result_diff = adfuller(Y_train_diff)

print(f'ADF statistics after differencing : {adf_result_diff[0]}')
print(f'p-value after differencing : {adf_result_diff[1]}')

ADF statistics after differencing : -5.878890182647569
p-value after differencing : 3.112089577251429e-07


In [42]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(Y_train_diff, lags=14)
plt.show()

plot_pacf(Y_train_diff, lags=14, method='ywm')
plt.show()

In [43]:
from statsmodels.tsa.arima.model import ARIMA

arima_order = [
    (1,1,1),
    (2,1,1),
    (1,1,2),
    (2,1,2)
]

arima_result = {}
arima_predictions = {}

for order in arima_order:
    fold_mae = []
    fold_predictions = []

    for train_fold, validation_fold in folds:
        train_values = train_fold['Children in HHS Care '].values
        validation_values = validation_fold['Children in HHS Care '].values

        # fit ARIMA only only on this folds trainning data
        model = ARIMA(train_values, order=order)
        model_fit = model.fit()

        # forcasting 14 validaton observation
        prediction = model_fit.forecast(steps = len(validation_values))

        # store prediction for later evaluation
        fold_predictions.append({
            'fold' : len(fold_predictions) + 1,
            'actual' : validation_values,
            'prediction' : prediction 
        })

        # calculate MAE 
        mae = mean_absolute_error(validation_values, prediction)
        fold_mae.append(mae)

    # average MAE across 10 folds
    arima_result[order] = sum(fold_mae) / len(fold_mae)

    # store prediction for this Arima order
    arima_predictions[order] = fold_predictions

    print(f'ARIMA{order} Average mae : {arima_result[order]:.2f}')

ARIMA(1, 1, 1) Average mae : 303.74
ARIMA(2, 1, 1) Average mae : 302.05
ARIMA(1, 1, 2) Average mae : 269.55


c:\Users\HP\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


ARIMA(2, 1, 2) Average mae : 268.53


In [44]:
from statsmodels.tsa.stattools import acf

acf_value = acf(Y_train_diff, nlags= 42, fft=True)

print('Lag 7 autocorrelation : ', acf_value[7])
print('Lag 14 autocorrelation : ', acf_value[14])

Lag 7 autocorrelation :  -0.07947174541303158
Lag 14 autocorrelation :  0.24332240147486292


In [45]:
plot_acf(Y_train_diff, lags=42)
plt.show()

In [46]:
print('lag 14 : ', acf_value[14])
print('lag 28 : ', acf_value[28])
print('lag 42 : ', acf_value[42])

lag 14 :  0.24332240147486292
lag 28 :  0.03022461391479279
lag 42 :  -0.04926865418830546


In [47]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_mae = []
sarimax_predictions = []

for train_fold, validation_fold in folds:
    train_values = train_fold['Children in HHS Care '].values
    validation_values = validation_fold['Children in HHS Care '].values

    # Fit Sarimax on this fold trainning data
    model = SARIMAX(
        train_values,
        order= (1, 1, 2),
        enforce_stationarity= False,
        enforce_invertibility= False
    )

    model_fit = model.fit(disp= False)

    # forcasting the validation period 
    prediction = model_fit.forecast(steps = len(validation_values))

    # sotre prediction for later evaluations
    sarimax_predictions.append({
        'fold' : fold_number,
        'actual' : validation_values,
        'prediction' : prediction
    })

    # calculate the MAE
    mae = mean_absolute_error(validation_values, prediction)
    sarimax_mae.append(mae)

average_sarimax_mae = sum(sarimax_mae)/len(sarimax_mae)

print('Average SARIMAX MAE : ', round(average_sarimax_mae, 2))

Average SARIMAX MAE :  248.69


Exponential Smoothing

In [48]:
from statsmodels.tsa.holtwinters import Holt

holt_mae = []
holt_predictions = []

for train_fold, validation_fold in folds:
    train_values = train_fold['Children in HHS Care '].values
    validation_values = validation_fold['Children in HHS Care '].values

    # fit Halt's exponential smoothing on the trainning fold
    model = Holt(train_values)
    model_fit = model.fit(optimized=True)

    # forecast the validation period 
    prediction = model_fit.forecast(steps = len(validation_values))

    #store the results for later evaluation
    holt_predictions.append({
        'fold' : fold_number,
        'actual' : validation_values,
        'prediction' : prediction
    })

    # calculate the MAE
    mae = mean_absolute_error(validation_values, prediction)
    holt_mae.append(mae)

# Average holt mae across all folds 
average_holt_mae = sum(holt_mae)/len(holt_mae)

print('Average Holt exponential MAE : ', round(average_holt_mae,2))

Average Holt exponential MAE :  275.34


# Machine learning Model

In [49]:
from sklearn.ensemble import RandomForestRegressor

rfr_mae = []
rf_predictions = []

for train_fold, validation_fold in folds:
    # Feature 
    X_train_fold = train_fold[feature]
    X_validation_fold = validation_fold[feature]

    # Target
    Y_train_fold = train_fold['Children in HHS Care ']
    Y_validation_fold = validation_fold['Children in HHS Care ']

    # create Random Forest model
    model = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )

    # Train model
    model.fit(X_train_fold, Y_train_fold)
    # Prediction validation period
    prediction = model.predict(X_validation_fold)

    # store prediction values for later evaluation
    rf_predictions.append({
        'fold' : fold_number,
        'actual' : Y_validation_fold.values,
        'prediction' : prediction
    })

    # calculate mae
    mae = mean_absolute_error(Y_validation_fold, prediction)
    rfr_mae.append(mae)

# Average MAE across the folds 
average_rf_mae = sum(rfr_mae)/len(rfr_mae)

print('Average Random forest MAE : ', round(average_rf_mae, 2))

Average Random forest MAE :  275.27


In [50]:
from sklearn.ensemble import GradientBoostingRegressor

gb_mae = []
gb_predictions = []

for train_fold, validation_fold in folds:
    #Feature 
    X_train_fold = train_fold[feature]
    X_validation_fold = validation_fold[feature]

    #Target
    Y_train_fold = train_fold['Children in HHS Care ']
    Y_validation_fold = validation_fold['Children in HHS Care ']

    # create Gradient Boosting MOdel
    model = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    )

    # train model
    model.fit(X_train_fold, Y_train_fold)
    # Prediction validation period 
    prediction = model.predict(X_validation_fold)

    # storing the predictions for later evaluation
    gb_predictions.append({
        'fold' : fold_number,
        'actual' : Y_validation_fold.values,
        'prediction' : prediction
    })

    # calculate mae
    mae = mean_absolute_error(Y_validation_fold, prediction)
    gb_mae.append(mae)

# Average mae across all folds
average_gb_mae = sum(gb_mae)/len(gb_mae)
print('Average Gradient Boosting MAE : ', round(average_gb_mae, 2))

Average Gradient Boosting MAE :  267.95


In [51]:
model_results = []

for result in naive_results:
    model_results.append({
        'model': 'Naive',
        'fold' : result['fold'],
        'actual' : result['actual'],
        'prediction': result['prediction']
    })

In [52]:
print(model_results)

[{'model': 'Naive', 'fold': 1, 'actual': array([6023, 6050, 6090, 5925, 5967, 6040, 6091, 6118, 5968, 6013, 6047,
       6075, 5889, 5935]), 'prediction': [np.int64(6002), np.int64(6023), np.int64(6050), np.int64(6090), np.int64(5925), np.int64(5967), np.int64(6040), np.int64(6091), np.int64(6118), np.int64(5968), np.int64(6013), np.int64(6047), np.int64(6075), np.int64(5889)]}, {'model': 'Naive', 'fold': 2, 'actual': array([6013, 6036, 6076, 5918, 5966, 6043, 6077, 5974, 6008, 6106, 6149,
       6148, 5990, 6007]), 'prediction': [np.int64(5935), np.int64(6013), np.int64(6036), np.int64(6076), np.int64(5918), np.int64(5966), np.int64(6043), np.int64(6077), np.int64(5974), np.int64(6008), np.int64(6106), np.int64(6149), np.int64(6148), np.int64(5990)]}, {'model': 'Naive', 'fold': 3, 'actual': array([6045, 6086, 6084, 5963, 5975, 5966, 6025, 5816, 5819, 5844, 5879,
       5850, 5664, 5715]), 'prediction': [np.int64(6007), np.int64(6045), np.int64(6086), np.int64(6084), np.int64(5963), np

In [53]:
for fold_number, (train_fold, validation_fold) in enumerate(folds, start=1):
    train_values = train_fold['Children in HHS Care '].values
    validation_values = validation_fold['Children in HHS Care '].values

    # generat moving avg prediction
    prediction = moving_average_forcasting(
        train_values,
        validation_values,
        window=7
    )

    # store prediction in common result structure 
    model_results.append({
        'model' : 'Moving_Average',
        'fold' : fold_number,
        'actual' : validation_values,
        'prediction' : prediction
    })

print('Moving average result added')
print('Total model result record : ', len(model_results))

Moving average result added
Total model result record :  22


In [54]:
from collections import Counter

Counter(result['model'] for result in model_results)

Counter({'Naive': 11, 'Moving_Average': 11})

In [55]:
for result in arima_predictions[(1,1,2)]:
    model_results.append({
        'model' : 'ARIMA(1,1,2)',
        'fold' : result['fold'],
        'actual' : result['actual'],
        'prediction' : result['prediction']
    })

print('Arima results added')
print('Total result : ', len(model_results))

Arima results added
Total result :  33


In [56]:
for result in sarimax_predictions:

    model_results.append({
        'model': 'SARIMA(1,1,2)(1,0,0,14)',
        'fold': result['fold'],
        'actual': result['actual'],
        'prediction': result['prediction']
    })

print("Total records:", len(model_results))
print(Counter(result['model'] for result in model_results))

Total records: 44
Counter({'Naive': 11, 'Moving_Average': 11, 'ARIMA(1,1,2)': 11, 'SARIMA(1,1,2)(1,0,0,14)': 11})


In [57]:
for result in holt_predictions:
    model_results.append({
        'model' : 'Holt Exponential Smoothing',
        'fold' : result['fold'],
        'actual' : result['actual'],
        'prediction' : result['prediction']
    })

print('Holt result added')
print('Total result : ', len(model_results))

Holt result added
Total result :  55


In [58]:
for result in rf_predictions:
    model_results.append({
        'model' : 'Random Forest',
        'fold' : result['fold'],
        'actual' : result['actual'],
        'prediction' : result['prediction']
    })

print('Random Forest result added')
print('Total result', len(model_results))

Random Forest result added
Total result 66


In [59]:
for result in gb_predictions:
    model_results.append({
        'model' : 'Gradient Boosting',
        'fold' : result['fold'],
        'actual' : result['actual'],
        'prediction' : result['prediction']
    })

print('Gradient Boosting result added')
print('Total result', len(model_results))

Gradient Boosting result added
Total result 77


In [60]:
seen = set()
clean_model_result = []

for result in model_results:
    key = (result['model'], result['fold'])

    if key not in seen:
        clean_model_result.append(result)
        seen.add(key)

model_results = clean_model_result
print('Total record', len(model_results))
print(Counter(result['model'] for result in model_results))

Total record 37
Counter({'Naive': 11, 'Moving_Average': 11, 'ARIMA(1,1,2)': 11, 'SARIMA(1,1,2)(1,0,0,14)': 1, 'Holt Exponential Smoothing': 1, 'Random Forest': 1, 'Gradient Boosting': 1})


In [61]:
print(len(sarimax_predictions))
print([result['fold'] for result in sarimax_predictions])

11
[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11]


In [62]:
print(len(holt_predictions))
print([result['fold'] for result in holt_predictions])

11
[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11]


In [63]:
print(len(rf_predictions))
print([result['fold'] for result in rf_predictions])

11
[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11]


In [64]:
print(len(gb_predictions))
print([result['fold'] for result in gb_predictions])

11
[11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11]


In [65]:
for predictions in [
    sarimax_predictions,
    holt_predictions,
    rf_predictions,
    gb_predictions
]:
    for fold_number, result in enumerate(predictions, start=1):
        result['fold'] = fold_number

In [66]:
print([result['fold'] for result in sarimax_predictions])
print([result['fold'] for result in holt_predictions])
print([result['fold'] for result in rf_predictions])
print([result['fold'] for result in gb_predictions])

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [67]:
print("Total records:", len(model_results))
print(Counter(result['model'] for result in model_results))

Total records: 37
Counter({'Naive': 11, 'Moving_Average': 11, 'ARIMA(1,1,2)': 11, 'SARIMA(1,1,2)(1,0,0,14)': 1, 'Holt Exponential Smoothing': 1, 'Random Forest': 1, 'Gradient Boosting': 1})


In [68]:
evaluation_results = []

for result in model_results:
    mae = mean_absolute_error(
        result['actual'],
        result['prediction']
    )

    evaluation_results.append({
        'model' : result['model'],
        'fold' : result['fold'],
        'MAE' : mae
    })

evaluation_df = pd.DataFrame(evaluation_results)

# Average MAE for each model
mae_comparison = (evaluation_df.groupby('model')['MAE'].mean().sort_values())
print(mae_comparison)

model
Gradient Boosting              50.035406
Naive                          58.805195
Random Forest                  59.776786
SARIMA(1,1,2)(1,0,0,14)       130.720330
Moving_Average                166.185529
ARIMA(1,1,2)                  269.546816
Holt Exponential Smoothing    320.273068
Name: MAE, dtype: float64


In [69]:
from sklearn.metrics import mean_squared_error
rmse_results = []

for result in model_results:
    rmse = np.sqrt(mean_squared_error(result['actual'], result['prediction']))

    rmse_results.append({
        'model' : result['model'],
        'fold' : result['fold'],
        'RMSE' : rmse
    })

rmse_df = pd.DataFrame(rmse_results)

# Average RMSE for each model
rmse_comparison = (
    rmse_df.groupby('model')['RMSE'].mean().sort_values()
)

print(rmse_comparison)

model
Gradient Boosting              76.424532
Random Forest                  77.841556
Naive                          82.063212
SARIMA(1,1,2)(1,0,0,14)       149.979940
Moving_Average                185.949095
ARIMA(1,1,2)                  305.348686
Holt Exponential Smoothing    357.545455
Name: RMSE, dtype: float64


In [70]:
from sklearn.metrics import mean_absolute_percentage_error

mape_results = []

for result in model_results:
    mape = mean_absolute_percentage_error(result['actual'], result['prediction']) * 100

    mape_results.append({
        'model' : result['model'],
        'fold' : result['fold'],
        'MAPE' : mape
    })

mape_df = pd.DataFrame(mape_results)

# Average mape for each model
mape_comparison = (
    mape_df.groupby('model')['MAPE'].mean().sort_values()
)
print(mape_comparison)

model
Gradient Boosting             0.831384
Random Forest                 0.992562
Naive                         1.305100
SARIMA(1,1,2)(1,0,0,14)       2.162538
Moving_Average                4.130332
Holt Exponential Smoothing    5.320142
ARIMA(1,1,2)                  6.666690
Name: MAPE, dtype: float64


In [71]:
horizon_results = []

for result in model_results:
    actual = result['actual']
    prediction = result['prediction']

    for horizon in [1,7,14]:
        horizon_mae = mean_absolute_error(
            actual[:horizon],
            prediction[:horizon]
        )

        horizon_results.append({
            'model' : result['model'],
            'fold' : result['fold'],
            'horizon' : horizon,
            'MAE' : horizon_mae
        })

horizon_df = pd.DataFrame(horizon_results)

horizon_comparison = (
    horizon_df.groupby(['model', 'horizon'])['MAE'].mean().reset_index()
)

print(horizon_comparison)

                         model  horizon         MAE
0                 ARIMA(1,1,2)        1   56.880313
1                 ARIMA(1,1,2)        7  170.014397
2                 ARIMA(1,1,2)       14  269.546816
3            Gradient Boosting        1    1.585510
4            Gradient Boosting        7   33.632339
5            Gradient Boosting       14   50.035406
6   Holt Exponential Smoothing        1   61.769743
7   Holt Exponential Smoothing        7  187.650398
8   Holt Exponential Smoothing       14  320.273068
9               Moving_Average        1  173.701299
10              Moving_Average        7  184.120594
11              Moving_Average       14  166.185529
12                       Naive        1   79.181818
13                       Naive        7   61.922078
14                       Naive       14   58.805195
15               Random Forest        1   46.600000
16               Random Forest        7   54.310000
17               Random Forest       14   59.776786
18     SARIM

# KPI's

In [72]:
# checking the capacity threshold

capacity_threshold = Y_train.quantile(0.95)
print('Analytical capacity threshold', round(capacity_threshold, 0))

Analytical capacity threshold 10841.0


In [73]:
capacity_breach_result = []

for result in model_results:
    prediction = np.array(result['prediction'])
    breach_count = np.sum(prediction > capacity_threshold)
    breach_rate = breach_count / len(prediction)

    capacity_breach_result.append({
        'model' : result['model'],
        'fold' : result['fold'],
        'breach_rate' : breach_rate * 100
    })

capacity_breach_df = pd.DataFrame(capacity_breach_result)

capacity_breach_comparison = (
    capacity_breach_df.groupby('model')['breach_rate'].mean().sort_values(ascending=False)
)
print(capacity_breach_comparison)

model
ARIMA(1,1,2)                  0.0
Gradient Boosting             0.0
Holt Exponential Smoothing    0.0
Moving_Average                0.0
Naive                         0.0
Random Forest                 0.0
SARIMA(1,1,2)(1,0,0,14)       0.0
Name: breach_rate, dtype: float64


In [74]:
print('capacity threshold', capacity_threshold)
print('Maximum actual Value', Y_train.max())

for result in model_results:
    print(
        result['model'],'Max prediction', np.max(result['prediction'])
    )

capacity threshold 10841.25
Maximum actual Value 11516
Naive Max prediction 6118
Naive Max prediction 6149
Naive Max prediction 6086
Naive Max prediction 6481
Naive Max prediction 6707
Naive Max prediction 5882
Naive Max prediction 4621
Naive Max prediction 2752
Naive Max prediction 2238
Naive Max prediction 2267
Naive Max prediction 2309
Moving_Average Max prediction 6170.857142857143
Moving_Average Max prediction 6072.142857142857
Moving_Average Max prediction 6075.857142857143
Moving_Average Max prediction 6332.285714285715
Moving_Average Max prediction 6622.714285714285
Moving_Average Max prediction 6413.428571428572
Moving_Average Max prediction 4730.857142857143
Moving_Average Max prediction 3013.285714285714
Moving_Average Max prediction 2329.8571428571427
Moving_Average Max prediction 2245.714285714286
Moving_Average Max prediction 2302.8571428571427
ARIMA(1,1,2) Max prediction 6019.946763777813
ARIMA(1,1,2) Max prediction 5976.445288941905
ARIMA(1,1,2) Max prediction 6022.5974

In [75]:
actual_breach_result = []

for result in model_results:
    actual = np.array(result['actual'])
    prediction = np.array(result['prediction'])

    actual_breach_rate = np.mean(actual > capacity_threshold) * 100
    forecast_breach_rate = np.mean(prediction > capacity_threshold) * 100

    actual_breach_result.append({
        'model' : result['model'],
        'fold' : result['fold'],
        'actual_breach_rate' : actual_breach_rate,
        'forecast_breach_rate' : forecast_breach_rate
    })

actual_breach_df = pd.DataFrame(actual_breach_result)

actual_breach_comaprison = (
    actual_breach_df.groupby('model')[['actual_breach_rate', 'forecast_breach_rate']].mean()
)
print(actual_breach_comaprison)

                            actual_breach_rate  forecast_breach_rate
model                                                               
ARIMA(1,1,2)                               0.0                   0.0
Gradient Boosting                          0.0                   0.0
Holt Exponential Smoothing                 0.0                   0.0
Moving_Average                             0.0                   0.0
Naive                                      0.0                   0.0
Random Forest                              0.0                   0.0
SARIMA(1,1,2)(1,0,0,14)                    0.0                   0.0


In [76]:
all_validation_actual = np.concatenate([np.array(result['actual']) for result in model_results])

print('Maximum validation actual', all_validation_actual.max())
print('Capacity threshold', capacity_threshold)

Maximum validation actual 6707
Capacity threshold 10841.25


In [77]:
forecast_accuracy = 100 - mape_comparison
print('forcast accuracy (%) : ')
print(forecast_accuracy)

forcast accuracy (%) : 
model
Gradient Boosting             99.168616
Random Forest                 99.007438
Naive                         98.694900
SARIMA(1,1,2)(1,0,0,14)       97.837462
Moving_Average                95.869668
Holt Exponential Smoothing    94.679858
ARIMA(1,1,2)                  93.333310
Name: MAPE, dtype: float64


In [78]:
surge_threshold = Y_train.quantile(0.90)
print('surge threshold', round(surge_threshold,0))

surge threshold 10263.0


In [79]:
stability_result = []

for result in model_results:
    prediction = np.array(result['prediction'])

    # Average absolute change between consecutive forecast
    forecast_change = np.mean(np.abs(np.diff(prediction)))

    stability_result.append({
        'model' : result['model'],
        'fold' : result['fold'],
        'forecast_change' : forecast_change
    })

stability_df = pd.DataFrame(stability_result)

stability_comparison = (
    stability_df.groupby('model')['forecast_change'].mean().sort_values()
)

print('Average forecast change')
print(stability_comparison)

Average forecast change
model
ARIMA(1,1,2)                  10.026515
SARIMA(1,1,2)(1,0,0,14)       12.208018
Holt Exponential Smoothing    40.769742
Moving_Average                41.557443
Naive                         57.741259
Gradient Boosting             72.001472
Random Forest                 75.324231
Name: forecast_change, dtype: float64


In [80]:
forecast_stability_index = (
    100 * (1 - stability_comparison / stability_comparison.max())
).round(2)

print('Forecast stability index (%) : ')
print(forecast_stability_index.sort_values(ascending= False))

Forecast stability index (%) : 
model
ARIMA(1,1,2)                  86.69
SARIMA(1,1,2)(1,0,0,14)       83.79
Holt Exponential Smoothing    45.87
Moving_Average                44.83
Naive                         23.34
Gradient Boosting              4.41
Random Forest                  0.00
Name: forecast_change, dtype: float64


In [81]:
print(
    evaluation_df.groupby('model')['MAE'].agg(['count', 'mean', 'std'])
)

                            count        mean         std
model                                                    
ARIMA(1,1,2)                   11  269.546816  366.286196
Gradient Boosting               1   50.035406         NaN
Holt Exponential Smoothing      1  320.273068         NaN
Moving_Average                 11  166.185529  167.524770
Naive                          11   58.805195   40.645457
Random Forest                   1   59.776786         NaN
SARIMA(1,1,2)(1,0,0,14)         1  130.720330         NaN


In [82]:
print(Counter(result['model'] for result in model_results))

Counter({'Naive': 11, 'Moving_Average': 11, 'ARIMA(1,1,2)': 11, 'SARIMA(1,1,2)(1,0,0,14)': 1, 'Holt Exponential Smoothing': 1, 'Random Forest': 1, 'Gradient Boosting': 1})


In [83]:
print("Naive folds:", len(naive_mae))
print("Moving Average folds:", len(moving_avg_mae))
print("SARIMA folds:", len(sarimax_mae))
print("Holt folds:", len(holt_mae))
print("Random Forest folds:", len(rfr_mae))
print("Gradient Boosting folds:", len(gb_mae))

Naive folds: 11
Moving Average folds: 11
SARIMA folds: 11
Holt folds: 11
Random Forest folds: 11
Gradient Boosting folds: 11


In [84]:
print("Number of folds:", len(folds))

Number of folds: 11


In [85]:
print(arima_result)

{(1, 1, 1): 303.74412162297835, (2, 1, 1): 302.053021081623, (1, 1, 2): 269.54681584959803, (2, 1, 2): 268.5274741315191}


In [86]:
arima_212_mae = []

for result in model_results:
    mae = mean_absolute_error(
        result['actual'], result['prediction']
    )

    arima_212_mae.append(mae)

print('Arima (2,1,2) fold MAE : ')
print(arima_212_mae)

print (
    'Average',
    round(sum(arima_212_mae) / len(arima_212_mae), 2)
)

Arima (2,1,2) fold MAE : 
[66.78571428571429, 65.14285714285714, 58.57142857142857, 65.92857142857143, 87.71428571428571, 104.35714285714286, 133.5, 37.285714285714285, 12.357142857142858, 10.142857142857142, 5.071428571428571, 80.81632653061219, 65.24489795918369, 85.73469387755098, 172.82653061224497, 173.3979591836734, 457.9081632653061, 511.9183673469389, 172.3775510204082, 58.08163265306117, 37.12244897959181, 12.612244897959174, 57.590079235497406, 79.78310230693094, 131.55840672833452, 469.4786002086081, 183.69997992005256, 748.8370101663975, 1135.9877484828326, 29.629808857522512, 37.76687366133735, 77.58818869991968, 13.09517607814548, 130.7203302380621, 320.2730676389133, 59.77678571428584, 50.03540619574206]
Average 162.18


In [87]:
print(len(fold_mae))
print(fold_mae)

11
[49.25247288220511, 73.81499920511578, 139.85434684195928, 420.3946625539514, 192.64654617675328, 772.1948992990954, 1146.0679651676728, 29.600943597450105, 39.62852161882701, 78.12748873525801, 12.219369368422283]


In [88]:
arima_robustness = 100 *(
    1 - np.std(fold_mae) / np.mean(fold_mae)
)

arima_robustness = max(0, round(arima_robustness, 2))

print('Arima(2,1,2) robustness score', arima_robustness)

Arima(2,1,2) robustness score 0


In [89]:
robustness_result = []

model_mae = [
    ('Naive', naive_mae),
    ('Moving Average', moving_avg_mae),
    ('ARIMA(2,1,2)', fold_mae),
    ('SARIMA', sarimax_mae),
    ('Holt Exponential Smoothing', holt_mae),
    ('Random Forest', rfr_mae),
    ('Gradient Boosting', gb_mae)
]

for model, mae_values in model_mae:
    mean_mae = np.mean(mae_values)
    std_mae = np.std(mae_values)

    cv = std_mae / mean_mae

    robustness_score = max(
        0, 100*(1-cv)
    )

    robustness_result.append({
        'model' : model,
        'mean_mae' : mean_mae,
        'std_mae' : std_mae,
        'CV' : cv,
        'robustness_score' : round(robustness_score, 2)
    })

robustness_df = pd.DataFrame(robustness_result)

robustness_comparison = (
    robustness_df.sort_values('robustness_score', ascending=False)
)

print(robustness_comparison)

                        model    mean_mae     std_mae        CV  \
0                       Naive   58.805195   38.753922  0.659022   
4  Holt Exponential Smoothing  275.338425  220.205403  0.799763   
1              Moving Average  166.185529  159.728601  0.961146   
2                ARIMA(2,1,2)  268.527474  352.466325  1.312589   
3                      SARIMA  248.693100  290.719420  1.168989   
5               Random Forest  275.273019  377.088605  1.369871   
6           Gradient Boosting  267.947674  378.008857  1.410756   

   robustness_score  
0             34.10  
4             20.02  
1              3.89  
2              0.00  
3              0.00  
5              0.00  
6              0.00  


Finnal KPI summary

In [90]:
kpi_summary = pd.DataFrame({
    'KPI' : [
        'Forecast Accuracy (%)',
        'Capacity Breach Rate (%)',
        'Surge Lead Time',
        'Forecast Stability Index',
        'Model Robustness'
    ],
    'Status / Result' : [
        'Available for all models',
        '0% during valdation',
        'N/A - no surge occured',
        'Available for all models',
        'Available for all models'
    ]
})

print(kpi_summary)

                        KPI           Status / Result
0     Forecast Accuracy (%)  Available for all models
1  Capacity Breach Rate (%)       0% during valdation
2           Surge Lead Time    N/A - no surge occured
3  Forecast Stability Index  Available for all models
4          Model Robustness  Available for all models


In [91]:
final_model_comparion = pd.concat(
    [
        mae_comparison.rename('MAE'),
        rmse_comparison.rename('RMSE'),
        mape_comparison.rename('MAPE')
    ],
    axis=1
)

final_model_comparion = final_model_comparion.sort_values('MAE')
print(final_model_comparion)

                                   MAE        RMSE      MAPE
model                                                       
Gradient Boosting            50.035406   76.424532  0.831384
Naive                        58.805195   82.063212  1.305100
Random Forest                59.776786   77.841556  0.992562
SARIMA(1,1,2)(1,0,0,14)     130.720330  149.979940  2.162538
Moving_Average              166.185529  185.949095  4.130332
ARIMA(1,1,2)                269.546816  305.348686  6.666690
Holt Exponential Smoothing  320.273068  357.545455  5.320142


In [92]:
horizon_table = horizon_comparison.pivot(index='model', columns='horizon', values= 'MAE')
horizon_table.columns = [
    'MAE_1_step',
    'MAE_7_step',
    "MAE_14_step"
]

horizon_table = horizon_table.sort_values('MAE_14_step')
print(horizon_table)

                            MAE_1_step  MAE_7_step  MAE_14_step
model                                                          
Gradient Boosting             1.585510   33.632339    50.035406
Naive                        79.181818   61.922078    58.805195
Random Forest                46.600000   54.310000    59.776786
SARIMA(1,1,2)(1,0,0,14)      42.803889   99.233695   130.720330
Moving_Average              173.701299  184.120594   166.185529
ARIMA(1,1,2)                 56.880313  170.014397   269.546816
Holt Exponential Smoothing   61.769743  187.650398   320.273068


In [93]:
robustness_final = robustness_df.set_index('model')['robustness_score'].rename({
    'ARIMA(2,1,2)': 'ARIMA(1,1,2)',
    'SARIMA': 'SARIMA(1,1,2)(1,0,0,14)'
})

final_selection = pd.concat(
    [
        mae_comparison.rename('MAE'),
        rmse_comparison.rename('RMSE'),
        mape_comparison.rename('MAPE'),
        horizon_table,
        robustness_final.rename('Robustness')
    ],
    axis=1
)

print(final_selection)

                                   MAE        RMSE      MAPE  MAE_1_step  \
model                                                                      
Gradient Boosting            50.035406   76.424532  0.831384    1.585510   
Naive                        58.805195   82.063212  1.305100   79.181818   
Random Forest                59.776786   77.841556  0.992562   46.600000   
SARIMA(1,1,2)(1,0,0,14)     130.720330  149.979940  2.162538   42.803889   
Moving_Average              166.185529  185.949095  4.130332  173.701299   
ARIMA(1,1,2)                269.546816  305.348686  6.666690   56.880313   
Holt Exponential Smoothing  320.273068  357.545455  5.320142   61.769743   
Moving Average                     NaN         NaN       NaN         NaN   

                            MAE_7_step  MAE_14_step  Robustness  
model                                                            
Gradient Boosting            33.632339    50.035406        0.00  
Naive                        61.922078   

In [94]:
final_selection.index = final_selection.index.str.replace(
    'Moving_Average',
    'Moving Average',
    regex=False
)

final_selection = final_selection[
    ~final_selection.index.duplicated(keep='first')
]

print(final_selection)

                                   MAE        RMSE      MAPE  MAE_1_step  \
model                                                                      
Gradient Boosting            50.035406   76.424532  0.831384    1.585510   
Naive                        58.805195   82.063212  1.305100   79.181818   
Random Forest                59.776786   77.841556  0.992562   46.600000   
SARIMA(1,1,2)(1,0,0,14)     130.720330  149.979940  2.162538   42.803889   
Moving Average              166.185529  185.949095  4.130332  173.701299   
ARIMA(1,1,2)                269.546816  305.348686  6.666690   56.880313   
Holt Exponential Smoothing  320.273068  357.545455  5.320142   61.769743   

                            MAE_7_step  MAE_14_step  Robustness  
model                                                            
Gradient Boosting            33.632339    50.035406        0.00  
Naive                        61.922078    58.805195       34.10  
Random Forest                54.310000    59.776786

## Final Model Selection

Based on the evaluated MAE, RMSE, MAPE, and multi-horizon performance, Gradient Boosting Regressor is selected as the primary forecasting model.

Gradient Boosting achieved the lowest overall MAE, RMSE, and MAPE and also demonstrated the lowest cumulative MAE at the 1-day, 7-day, and 14-day forecast horizons.

The Naïve persistence model remained a strong baseline and provides an important benchmark for evaluating whether the machine-learning model adds predictive value.

Although Gradient Boosting showed lower robustness under the project-defined fold-consistency score, its superior predictive accuracy makes it the preferred model for the forecasting application.

In [95]:
# Final data training 
X_train_final = train_df[feature]
Y_train_final = train_df['Children in HHS Care ']

# test data
X_test_final = test_df[feature]

# create final Gradient Boosting model
final_gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

# train on complete trainning data 
final_gb_model.fit(X_train_final, Y_train_final)

# forecast the test period
final_gb_prediction = final_gb_model.predict(X_test_final)

print('Number of forecast : ', len(final_gb_prediction))
print('1st 10 forecats : ')
print(final_gb_prediction[:10])

Number of forecast :  142
1st 10 forecats : 
[2382.75743365 2397.50248384 2405.63252241 2408.86410572 2408.57134787
 2408.32000001 2410.63202243 2395.17988811 2394.82750717 2408.57134787]


In [96]:
forecast_df = pd.DataFrame({
    'Date' : test_df['Date'].values,
    'Actual' : test_df['Children in HHS Care '].values,
    'Forecast' : final_gb_prediction
})

print(forecast_df.head())

        Date  Actual     Forecast
0 2025-05-18    2437  2382.757434
1 2025-05-19    2457  2397.502484
2 2025-05-21    2467  2405.632522
3 2025-05-22    2473  2408.864106
4 2025-05-27    2504  2408.571348


In [97]:
# calculating validation residuals for Gradient Boosting
gb_residuals = []

for result in gb_predictions:
    actual = np.array(result['actual'])
    prediction = np.array(result['prediction'])

    residuals = actual - prediction
    gb_residuals.extend(residuals)

# estimate residual standard deviation
residual_std = np.std(gb_residuals, ddof=1)

# 95% approximate prediction interval
forecast_df['Lower_95'] = (
    forecast_df['Forecast'] - 1.96 * residual_std
).clip(lower=0)

forecast_df['Upper_95'] = (
    forecast_df['Forecast'] + 1.96 * residual_std
)

print('Residual standar deviation', round(residual_std, 2))
print(forecast_df.head())

Residual standar deviation 450.29
        Date  Actual     Forecast     Lower_95     Upper_95
0 2025-05-18    2437  2382.757434  1500.193047  3265.321820
1 2025-05-19    2457  2397.502484  1514.938097  3280.066870
2 2025-05-21    2467  2405.632522  1523.068136  3288.196909
3 2025-05-22    2473  2408.864106  1526.299719  3291.428492
4 2025-05-27    2504  2408.571348  1526.006961  3291.135734


In [98]:
plt.figure(figsize=(14,6))
plt.plot(forecast_df['Date'], forecast_df['Actual'], label='Actual')
plt.plot(forecast_df['Date'], forecast_df['Forecast'], label='Forecast')
plt.fill_between(forecast_df['Date'], forecast_df['Lower_95'], forecast_df['Upper_95'], alpha= 0.2, label='95% prediction interval')
plt.xlabel('Date')
plt.ylabel('Children in HHS Care')
plt.title('HHS Care Load Forecast with 95% Predictive Interval')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [99]:
# Discharge Demand Forecast

discharge_train = df.loc[
    train_df.index,
    'Children discharged from HHS Care'
].values

discharge_test = df.loc[
    test_df.index,
    'Children discharged from HHS Care'
].values

history = list(discharge_train)
discharge_prediction = []

# 7-day moving average forecast
for _ in discharge_test:

    prediction = np.mean(history[-7:])

    discharge_prediction.append(prediction)

    # Add prediction to history for next forecast
    history.append(prediction)

# Create discharge forecast dataframe
discharge_forecast_df = pd.DataFrame({
    'Date': test_df['Date'].values,
    'Actual_Discharge': discharge_test,
    'Forecast_Discharge': discharge_prediction
})

print(discharge_forecast_df.head())

        Date  Actual_Discharge  Forecast_Discharge
0 2025-05-18               9.0            9.571429
1 2025-05-19               6.0            9.510204
2 2025-05-21              18.0            9.297376
3 2025-05-22              15.0           10.197001
4 2025-05-27              11.0           10.653716


In [100]:
plt.figure(figsize=(14, 6))

plt.plot(
    discharge_forecast_df['Date'],
    discharge_forecast_df['Actual_Discharge'],
    label='Actual Discharges'
)

plt.plot(
    discharge_forecast_df['Date'],
    discharge_forecast_df['Forecast_Discharge'],
    label='Forecast Discharges'
)

plt.xlabel('Date')
plt.ylabel('Children Discharged')
plt.title('HHS Discharge Demand Forecast')

plt.legend()
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [101]:
discharge_mae = mean_absolute_error(
    discharge_forecast_df['Actual_Discharge'],
    discharge_forecast_df['Forecast_Discharge']
)

discharge_rmse = np.sqrt(
    mean_squared_error(
        discharge_forecast_df['Actual_Discharge'],
        discharge_forecast_df['Forecast_Discharge']
    )
)

print("Discharge Forecast MAE:", round(discharge_mae, 2))
print("Discharge Forecast RMSE:", round(discharge_rmse, 2))

Discharge Forecast MAE: 4.81
Discharge Forecast RMSE: 6.79


In [102]:
forecast_df.to_csv('forecast_output.csv', index=False)
discharge_forecast_df.to_csv('discharge_forecast_output.csv', index=False)

print('Forecast files saved')

Forecast files saved


In [103]:
final_selection.to_csv(
    'model_comparison.csv',
    index=True
)

print("Model comparison saved.")

Model comparison saved.
